# CKAN Dataset Registration Workflow

This notebook demonstrates the proper workflow to register your analyzed dataset in CKAN using the CKAN Registration Agent.

**Key concept**: To register a dataset, you must first **analyze** it with either:
- A readable file path (`upload_dir`)
- A source URL (`source_url`)
- Explicit dataset metadata (`dataset` with title, name, notes)

This tells the agent what data you want to register and generates a metadata proposal.


## 1. Import Required Libraries

Import necessary libraries for working with datasets and CKAN metadata.


In [ ]:
import json
import os
from pathlib import Path
import pandas as pd
import requests

# CKAN Registration Agent client
# If you have the agent API running, you can import the client
# For now, we'll use requests to call the HTTP API
AGENT_BASE_URL = os.getenv("CKAN_AGENT_API_URL", "http://localhost:8787")


## 2. Load and Inspect Your Dataset

Load your analyzed dataset file and examine its structure.


In [ ]:
# Path to your dataset file
dataset_path = Path("folium_mapping.txt")  # Update with your actual file path

# Check if file exists
if not dataset_path.exists():
    print(f"⚠️  File not found: {dataset_path}")
    print("Please update the dataset_path variable with the correct path to your data file.")
else:
    print(f"✓ Found dataset: {dataset_path}")
    print(f"  Size: {dataset_path.stat().st_size / 1024:.2f} KB")
    
    # Preview first few lines
    with open(dataset_path, 'r') as f:
        lines = f.readlines()[:10]
    print(f"\n  First {len(lines)} lines:")
    for i, line in enumerate(lines, 1):
        print(f"    {i}: {line.rstrip()[:80]}")


## 3. Extract Dataset Metadata

Extract key metadata about your dataset to help the CKAN agent understand what you're registering.


In [ ]:
# Define metadata for your dataset
# ⚠️  ALL of these fields are REQUIRED by the CKAN agent
# If you don't provide them, the agent will ask for them explicitly!

dataset_metadata = {
    "title": "Folium Mapping Dataset",  # REQUIRED: Human-readable title
    "name": "folium-mapping-dataset",   # REQUIRED: CKAN-friendly slug (lowercase, hyphens, no spaces)
    "notes": "Dataset created using Folium for interactive mapping. Generated from Jupyter notebook analysis.",  # REQUIRED: Description
    "author": "Your Name",              # REQUIRED: Replace with your name
    "author_email": "your.email@example.com",  # REQUIRED: Replace with your email
    "maintainer": "Your Name",          # REQUIRED: Replace with your name
    "maintainer_email": "your.email@example.com",  # REQUIRED: Replace with your email
    "license_id": "cc-by",              # REQUIRED: CKAN license identifier
    "tags": [
        {"name": "folium"},
        {"name": "mapping"},
        {"name": "geospatial"},
        {"name": "jupyter"}
    ],
}

print("Dataset metadata template:")
print(json.dumps(dataset_metadata, indent=2))
print("\n⚠️  UPDATE THE VALUES ABOVE:")
print("  - title: A descriptive name for your dataset")
print("  - name: URL-friendly identifier (lowercase, hyphens only)")
print("  - notes: 1-2 sentence description")
print("  - author/maintainer: Your name")
print("  - author_email/maintainer_email: Your email address")


## 4. Submit Dataset for Analysis via CKAN Agent

Now submit your dataset to the CKAN Registration Agent. This triggers the **analyze** action which:
1. Reads your data file
2. Generates a metadata proposal
3. Creates a local registration session (thread)

This is the key step you were missing—the agent needs the `upload_dir` or file path to analyze!


In [ ]:
# ============================================================================
# SCENARIO 1: What happens if you submit WITHOUT required metadata?
# ============================================================================

incomplete_request = {
    "action": "analyze",
    "message": "Please analyze my dataset.",
    "upload_dir": str(dataset_path.parent),
    # ⚠️  NOTE: Missing dataset metadata!
}

print("=" * 80)
print("SCENARIO 1: Submitting WITHOUT required metadata")
print("=" * 80)
print("\nIncomplete request (missing dataset metadata):")
print(json.dumps(incomplete_request, indent=2, default=str))
print("\n⏸️  If you submit this, the CKAN agent will INTERRUPT and ask:")
print("   'I need at least one: readable local file path, upload directory,")
print("    source URL, or explicit dataset metadata (title, name, notes).'")
print("\nThe agent WON'T proceed until you provide these required fields.")

print("\n" + "=" * 80)
print("SCENARIO 2: Submitting WITH complete required metadata ✓")
print("=" * 80)

# Correct request with all required fields
analysis_request = {
    "action": "analyze",
    "message": "Please analyze my Folium mapping dataset and generate CKAN metadata.",
    "upload_dir": str(dataset_path.parent),  # Directory containing the data file
    "dataset": dataset_metadata,  # ✓ REQUIRED: All required fields present
}

print("\nComplete request (all required fields):")
print(json.dumps(analysis_request, indent=2, default=str))

print("\nSubmitting analysis request to CKAN Agent...")

# Call the CKAN Registration Agent API
try:
    response = requests.post(
        f"{AGENT_BASE_URL}/agent/ckan-registration/run",
        json=analysis_request,
        headers={
            "Content-Type": "application/json",
            # Add CKAN credentials if needed:
            # "x-ckan-api-token": "your-api-token",
            # "x-openai-api-key": "your-openai-key",
        },
        timeout=120,
    )
    response.raise_for_status()
    result = response.json()
    
    print("\n✓ Analysis submitted successfully!")
    print(f"  Thread ID (session_id): {result.get('thread_id')}")
    print(f"  Status: {result.get('status')}")
    
    # Check if agent is asking for clarification
    requires_action = result.get('requires_action')
    if requires_action and requires_action.get('type') == 'clarification_required':
        print(f"\n⚠️  Agent asking for clarification:")
        print(f"   {requires_action.get('message')}")
        print(f"   Required fields: {requires_action.get('required_fields')}")
    else:
        # Save the thread ID for later steps
        session_id = result.get('thread_id')
        print(f"\n📌 Save this session_id for the next steps: {session_id}")
    
except requests.exceptions.RequestException as e:
    print(f"\n✗ Error calling CKAN Agent: {e}")
    print("Make sure the CKAN Registration Agent API is running at:", AGENT_BASE_URL)


## 5. Review Generated Metadata (Dry Run)

After analysis, run a **dry-run** to see the proposed CKAN metadata without actually registering it.


In [ ]:
# Run a dry-run to preview the metadata WITHOUT applying changes to CKAN
dry_run_request = {
    "action": "dry-run",
    "session_id": session_id,  # Use the session from the previous analysis
    "message": "Show me the proposed CKAN metadata and resources before registration.",
}

print("Running dry-run to preview metadata...")

try:
    response = requests.post(
        f"{AGENT_BASE_URL}/agent/ckan-registration/run",
        json=dry_run_request,
        headers={"Content-Type": "application/json"},
        timeout=120,
    )
    response.raise_for_status()
    result = response.json()
    
    print("\n✓ Dry-run completed successfully!")
    print("\nProposed CKAN Metadata:")
    print(json.dumps(result.get('result', {}), indent=2, default=str))
    
    print("\n" + "="*80)
    print("REVIEW THIS CAREFULLY before approving!")
    print("="*80)
    
except requests.exceptions.RequestException as e:
    print(f"\n✗ Error running dry-run: {e}")


## 6. Approve and Register in CKAN

Once you've reviewed the dry-run output and are satisfied, register the dataset by sending the approval.

⚠️  **Important**: The approval must be exactly **`REGISTER`** (uppercase) to proceed.


In [ ]:
# IMPORTANT: Change this to True only after reviewing the dry-run output above!
READY_TO_REGISTER = False  # Set to True to proceed with registration

if READY_TO_REGISTER:
    # Submit approval to register the dataset in CKAN
    approval_request = {
        "action": "apply",
        "session_id": session_id,
        "approval": "REGISTER",  # Must be exactly this value
        "message": "Approved. Register this dataset in CKAN.",
    }
    
    print("⚠️  Submitting registration approval to CKAN...")
    print("This will create/update the dataset in your CKAN instance.")
    
    try:
        response = requests.post(
            f"{AGENT_BASE_URL}/agent/ckan-registration/run",
            json=approval_request,
            headers={"Content-Type": "application/json"},
            timeout=120,
        )
        response.raise_for_status()
        result = response.json()
        
        print("\n✓ Dataset registered successfully in CKAN!")
        print(f"  Status: {result.get('status')}")
        print(f"  CKAN ID: {result.get('result', {}).get('id', 'N/A')}")
        print(f"  CKAN URL: {result.get('result', {}).get('url', 'N/A')}")
        
    except requests.exceptions.RequestException as e:
        print(f"\n✗ Error during registration: {e}")
else:
    print("⏸️  Registration approval SKIPPED.")
    print("\nTo register, follow these steps:")
    print("1. Review the dry-run output above")
    print("2. If satisfied, change READY_TO_REGISTER = False to READY_TO_REGISTER = True")
    print("3. Run this cell again")


## Summary: The Workflow

The **key point** you were missing: The CKAN Agent requires **input data** to analyze before creating metadata.

### Correct Workflow:

1. **ANALYZE** ✓ (Requires: `upload_dir`, `source_url`, or `dataset` metadata)
   - Agent reads your file(s)
   - Creates metadata proposal
   - Saves session (session_id)

2. **DRY-RUN** ✓ (Requires: `session_id` from analyze step)
   - Preview proposed metadata
   - Review without applying changes

3. **APPLY (Approval)** ✓ (Requires: `session_id` + `approval: "REGISTER"`)
   - Actually create/update CKAN dataset
   - Resources registered

### Common Mistake ❌

Asking to "create metadata" without providing:
- The file path (`upload_dir`)
- The data URL (`source_url`)
- Explicit dataset details (`dataset` object)

This is why you got: *"I need an existing analyzed CKAN thread before I can do that."*

The agent was trying to REVISE instead of ANALYZE because no data input was provided.

### Next Steps

1. Update `dataset_path` in cell 2 to point to your `folium_mapping.txt`
2. Update `dataset_metadata` in cell 3 with your dataset info
3. Run cells in order: analyze → dry-run → approval
4. Review each step before proceeding
